# 99 — Reproduce All

**This is the main entry point for reviewers.** It orchestrates the
notebooks/modules already defined elsewhere in this repository -- it does
NOT duplicate any algorithm. Set the two controls below, then run all
cells.

| RUN_MODE | DATA_MODE | Behavior |
|---|---|---|
| quick | public | Smoke test: load de-identified artifacts, run integrity audit, reproduce a subset of statistics, regenerate representative tables. No heavy optimization. |
| full | public | Reproduce the complete set of publicly-reproducible analytical results (SA calibration summaries, held-out summaries, fuzzy analysis, Liu-ALNS comparison, ERR results, daily-instance inference, manuscript tables/figures) from de-identified numerical artifacts. |
| quick | private | Computational smoke test of the routing pipeline using authorized operational inputs. |
| full | private | Complete computational reproduction: preprocessing -> quantile calibration -> simulator -> NR/FR/RG/SA -> calibration -> held-out experiment -> fuzzy -> Liu-ALNS -> ERR -> statistics -> tables. |


In [ ]:
# --- SET THESE TWO CONTROLS, THEN RUN ALL CELLS ---
RUN_MODE = "quick"    # "quick" or "full"
DATA_MODE = "public"  # "public" or "private"

assert RUN_MODE in ("quick", "full")
assert DATA_MODE in ("public", "private")
print(f"RUN_MODE={RUN_MODE}, DATA_MODE={DATA_MODE}")


## What this notebook does

It runs, in order, the SAME cells that live in notebooks 00-11, by
executing those notebook files programmatically (so there is exactly one
copy of every algorithm and every cell -- in the individual numbered
notebooks -- and this file is a thin sequencer over them).


In [ ]:
import os, sys, json, subprocess

# --- Configure these two if running on a fresh Colab runtime ---
# REPO_URL: fill this in with the GitHub URL once the repository has been
# published (e.g. "https://github.com/<org>/IJIES_Reproducible_Routing.git").
# Left empty here because no public URL exists yet at packaging time.
REPO_URL = ""
REPO_DIRNAME = "IJIES_Reproducible_Routing"

def in_colab():
    try:
        import google.colab  # noqa
        return True
    except ImportError:
        return False

IN_COLAB = in_colab()
print(f"Running in Colab: {IN_COLAB}")

def looks_like_repo_root(path):
    """A directory is considered a valid checkout if it has the src/data.py
    marker file -- cheap, specific, and avoids false positives on an empty
    or unrelated folder."""
    return os.path.exists(os.path.join(path, "src", "data.py"))

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_PARENT = "/content/drive/MyDrive/IJIES_Routing_Reproducibility"
    os.makedirs(DRIVE_PARENT, exist_ok=True)

    # 1) Already checked out directly at DRIVE_PARENT?
    if looks_like_repo_root(DRIVE_PARENT):
        REPO_ROOT = DRIVE_PARENT
    # 2) Checked out as a subfolder of DRIVE_PARENT (e.g. after a manual
    #    upload or a previous clone)?
    elif looks_like_repo_root(os.path.join(DRIVE_PARENT, REPO_DIRNAME)):
        REPO_ROOT = os.path.join(DRIVE_PARENT, REPO_DIRNAME)
    # 3) Not found -- clone it, if a URL has been configured.
    elif REPO_URL:
        target = os.path.join(DRIVE_PARENT, REPO_DIRNAME)
        print(f"No existing checkout found; cloning {REPO_URL} into {target} ...")
        subprocess.run(["git", "clone", REPO_URL, target], check=True)
        REPO_ROOT = target
    else:
        raise FileNotFoundError(
            "No repository checkout found under Google Drive, and REPO_URL is "
            "not set. Either (a) set REPO_URL above to this repository's GitHub "
            "URL once it has been published and re-run this cell, or (b) "
            "manually upload/clone the repository into "
            f"'{DRIVE_PARENT}' (or a subfolder of it) before running this "
            "notebook.")
    PROJECT_ROOT = DRIVE_PARENT
else:
    # Local execution: assume this notebook lives at <repo>/notebooks/.
    candidate = os.path.abspath(os.path.join(os.getcwd(), ".."))
    if looks_like_repo_root(candidate):
        REPO_ROOT = candidate
    else:
        raise FileNotFoundError(
            f"'{candidate}' does not look like a repository checkout (no "
            "src/data.py found). Run this notebook from within the "
            "repository's notebooks/ folder, or edit REPO_ROOT manually.")
    PROJECT_ROOT = REPO_ROOT

sys.path.insert(0, REPO_ROOT)
print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"REPO_ROOT = {REPO_ROOT}")

NOTEBOOKS_DIR = os.path.join(REPO_ROOT, "notebooks")
NOTEBOOK_SEQUENCE = [
    "00_SETUP_AND_DATA_AUDIT", "01_PREPROCESSING", "02_TOMTOM_QUANTILE_CALIBRATION",
    "03_EXECUTION_SIMULATOR", "04_NR_FR_RG_SA_ENGINE", "05_SA_CALIBRATION",
    "06_MAIN_51DATE_EXPERIMENT", "07_FUZZY_DECISION_GATE", "08_LIU_ALNS_COMPARATOR",
    "09_TOMTOM_EMPIRICAL_ROBUSTNESS", "10_STATISTICAL_ANALYSIS", "11_GENERATE_TABLES_FIGURES",
]


## Sequencer: execute each notebook's code cells in this same kernel

In [ ]:
def run_notebook_cells(nb_name, shared_globals):
    path = os.path.join(NOTEBOOKS_DIR, f"{nb_name}.ipynb")
    with open(path) as f:
        nb_json = json.load(f)
    for cell in nb_json['cells']:
        if cell['cell_type'] != 'code':
            continue
        src = ''.join(cell['source'])
        if not src.strip():
            continue
        exec(compile(src, nb_name, 'exec'), shared_globals)

# Save the ORIGINALLY REQUESTED mode before the sequencer runs, so we can
# hard-verify it was not silently overwritten by any downstream notebook
# (this was a real bug: notebook 00's own standalone-default cell used to
# unconditionally reset RUN_MODE/DATA_MODE when sequenced from here).
REQUESTED_RUN_MODE = RUN_MODE
REQUESTED_DATA_MODE = DATA_MODE

shared_globals = {'__name__': '__main__', 'RUN_MODE': RUN_MODE, 'DATA_MODE': DATA_MODE}
status_log = {}
for nb_name in NOTEBOOK_SEQUENCE:
    print(f"\n{'='*70}\nRunning {nb_name}\n{'='*70}")
    try:
        run_notebook_cells(nb_name, shared_globals)
        status_key = f"NOTEBOOK_{nb_name[:2]}_STATUS"
        status_log[nb_name] = shared_globals.get(status_key, "UNKNOWN")
        print(f"{nb_name}: {status_log[nb_name]}")
    except Exception as e:
        status_log[nb_name] = f"FAILED: {type(e).__name__}: {e}"
        print(f"{nb_name}: FAILED -- {e}")
        raise

    # Hard assertion: the active mode inside shared_globals must still match
    # what the user requested, after EVERY notebook in the sequence.
    active_run_mode = shared_globals.get('RUN_MODE')
    active_data_mode = shared_globals.get('DATA_MODE')
    if active_run_mode != REQUESTED_RUN_MODE or active_data_mode != REQUESTED_DATA_MODE:
        raise AssertionError(
            f"MODE PROPAGATION FAILURE after {nb_name}: requested "
            f"RUN_MODE={REQUESTED_RUN_MODE}/DATA_MODE={REQUESTED_DATA_MODE} but active "
            f"RUN_MODE={active_run_mode}/DATA_MODE={active_data_mode}. A downstream "
            f"notebook silently changed the mode -- this must never happen.")

print(f"\nMode propagation verified intact through all {len(NOTEBOOK_SEQUENCE)} notebooks: "
      f"RUN_MODE={REQUESTED_RUN_MODE}, DATA_MODE={REQUESTED_DATA_MODE}")


## Final summary

In [ ]:
print("="*70)
print("REPRODUCE-ALL SUMMARY")
print("="*70)
for nb_name, status in status_log.items():
    print(f"{status:8s}  {nb_name}")

all_pass = all(str(v) == "PASS" for v in status_log.values())
print(f"\nOVERALL: {'ALL PASS' if all_pass else 'ONE OR MORE STAGES FAILED'}")

results_dir = os.path.join(REPO_ROOT, "results")
os.makedirs(results_dir, exist_ok=True)
with open(os.path.join(results_dir, "reproduce_all_status.json"), "w") as f:
    json.dump({"run_mode": RUN_MODE, "data_mode": DATA_MODE, "status_log": status_log,
               "overall": "ALL PASS" if all_pass else "FAILED"}, f, indent=2)
print(f"\nSaved results/reproduce_all_status.json")
print(f"\nGenerated tables: {os.path.join(REPO_ROOT, 'tables')}")
print(f"Generated figures: {os.path.join(REPO_ROOT, 'figures')}")
